# Notebook 11 — Stateful vs. Stateless

**Continual Learning in Context: RML Extension for CL-BENCH**

Notebook 00 introduced the workflow.  
Notebook 01 measured gain.  
Notebook 07 mapped latent structure.

Notebook 11 returns to the central CL-BENCH comparison:

> Does accumulated state help more than a fresh stateless run?

This notebook analyzes the stateful/stateless gap, identifies where state helps most, and flags where state may become risky.

Outputs:

- `results/11_state_vs_stateless_summary.json`
- `results/11_state_vs_stateless_instances.csv`
- `results/11_state_vs_stateless_by_variant.csv`
- `figures/11_reward_gap.png`
- `figures/11_state_advantage_by_variant.png`
- `figures/11_state_help_hurt_map.png`
- `results/11_state_vs_stateless_artifacts.zip`

## 0. Bootstrap Repo Path

This cell works locally or in Colab.

If opened directly in Colab and the repo is missing, it clones:

```text
https://github.com/thinkthoughts/continual-learning-bench-rml
```

In [ ]:
from pathlib import Path
import sys
import os
import subprocess

REPO_URL = "https://github.com/thinkthoughts/continual-learning-bench-rml.git"
REPO_NAME = "continual-learning-bench-rml"

def running_in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False

def find_rml_root(start: Path) -> Path | None:
    start = start.resolve()
    for base in [start, *start.parents]:
        if (base / "src" / "gain.py").exists():
            return base
        if (base / "rml" / "src" / "gain.py").exists():
            return base / "rml"
        if (base / REPO_NAME / "rml" / "src" / "gain.py").exists():
            return base / REPO_NAME / "rml"
    return None

cwd = Path.cwd().resolve()
RML_ROOT = find_rml_root(cwd)

if RML_ROOT is None and running_in_colab():
    repo_path = Path("/content") / REPO_NAME

    if repo_path.exists():
        print("Repo already exists. Pulling latest changes...")
        subprocess.run(["git", "-C", str(repo_path), "pull"], check=False)
    else:
        print("Cloning repo...")
        subprocess.run(["git", "clone", REPO_URL, str(repo_path)], check=True)

    RML_ROOT = repo_path / "rml"
    os.chdir(RML_ROOT)

elif RML_ROOT is not None:
    os.chdir(RML_ROOT)

else:
    raise RuntimeError(
        "Could not locate rml/src/gain.py. "
        "Run this notebook inside the repo, or in Colab allow the bootstrap cell to clone the repo."
    )

if str(RML_ROOT) not in sys.path:
    sys.path.insert(0, str(RML_ROOT))

RESULTS_DIR = RML_ROOT / "results"
FIGURES_DIR = RML_ROOT / "figures"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Running in Colab:", running_in_colab())
print("Current working directory:", Path.cwd())
print("RML root:", RML_ROOT)
print("results:", RESULTS_DIR)
print("figures:", FIGURES_DIR)

## 1. Imports

In [ ]:
import json
import zipfile
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.gain import compute_gain

print("Imports complete.")

## 2. Load Prior Trajectory

Notebook 11 prefers the trajectory from Notebook 01:

```text
results/01_gain_signal_trajectory.csv
```

Fallback order:

1. `results/01_gain_signal_trajectory.csv`
2. `results/00_context_gain.csv`
3. synthetic toy trajectory

In [ ]:
trajectory_01 = RESULTS_DIR / "01_gain_signal_trajectory.csv"
trajectory_00 = RESULTS_DIR / "00_context_gain.csv"

if trajectory_01.exists():
    df = pd.read_csv(trajectory_01)
    source_file = trajectory_01
    print("Loaded:", trajectory_01)
elif trajectory_00.exists():
    df = pd.read_csv(trajectory_00)
    source_file = trajectory_00
    print("Loaded:", trajectory_00)
else:
    print("No prior trajectory found. Creating fallback toy trajectory.")
    df = pd.DataFrame({
        "instance": np.arange(1, 13),
        "variant": [
            "A", "A", "A", "A",
            "B", "B", "B", "B",
            "C", "C", "C", "C",
        ],
        "stateful_reward": [
            0.18, 0.22, 0.28, 0.35,
            0.43, 0.48, 0.46, 0.52,
            0.40, 0.45, 0.55, 0.60,
        ],
        "stateless_reward": [
            0.18, 0.19, 0.20, 0.21,
            0.22, 0.20, 0.21, 0.22,
            0.23, 0.22, 0.24, 0.25,
        ],
    })
    source_file = None

if "gain" not in df.columns:
    df["gain"] = compute_gain(
        df["stateful_reward"].tolist(),
        df["stateless_reward"].tolist(),
    )

df["state_advantage"] = df["gain"]
df["state_advantage_ratio"] = np.where(
    df["stateless_reward"] != 0,
    df["stateful_reward"] / df["stateless_reward"],
    np.nan,
)

df["state_helped"] = df["state_advantage"] > 0
df["state_hurt"] = df["state_advantage"] < 0
df["state_neutral"] = df["state_advantage"] == 0

df

## 3. Stateful vs. Stateless Framing

CL-BENCH isolates continual learning by comparing the same system in two modes:

- **stateful**: prior experience is available
- **stateless**: each instance is solved from a fresh reset

The important question is not merely whether stateful reward is high.

The question is:

\[
r_t^{sf} - r_t^{sl}
\]

That difference measures whether accumulated state contributed useful context.

## 4. Instance-Level State Advantage

In [ ]:
instance_view = df[[
    "instance",
    "variant",
    "stateful_reward",
    "stateless_reward",
    "state_advantage",
    "state_advantage_ratio",
    "state_helped",
    "state_hurt",
    "state_neutral",
]].copy()

instance_view

## 5. Variant-Level State Advantage

This table shows where accumulated state was most useful.

In [ ]:
variant_view = (
    df.groupby("variant")
    .agg(
        instances=("instance", "count"),
        mean_stateful_reward=("stateful_reward", "mean"),
        mean_stateless_reward=("stateless_reward", "mean"),
        mean_state_advantage=("state_advantage", "mean"),
        total_state_advantage=("state_advantage", "sum"),
        helped_instances=("state_helped", "sum"),
        hurt_instances=("state_hurt", "sum"),
        neutral_instances=("state_neutral", "sum"),
    )
    .reset_index()
)

variant_view["help_rate"] = variant_view["helped_instances"] / variant_view["instances"]
variant_view

## 6. Figure — Reward Gap

This figure overlays stateful and stateless reward and fills the visible gap.

The gap is the state advantage.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(df["instance"], df["stateful_reward"], marker="o", label="Stateful")
ax.plot(df["instance"], df["stateless_reward"], marker="o", linestyle="--", label="Stateless")

ax.fill_between(
    df["instance"],
    df["stateless_reward"],
    df["stateful_reward"],
    alpha=0.2,
)

for boundary in df.groupby("variant")["instance"].min().tolist()[1:]:
    ax.axvline(boundary - 0.5, linestyle=":", linewidth=1)

ax.set_title("Notebook 11: Stateful vs. Stateless Reward Gap")
ax.set_xlabel("Instance")
ax.set_ylabel("Reward")
ax.legend()
ax.grid(True, alpha=0.3)

reward_gap_path = FIGURES_DIR / "11_reward_gap.png"
fig.tight_layout()
fig.savefig(reward_gap_path, dpi=160)

reward_gap_path

## 7. Figure — State Advantage by Variant

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.bar(
    variant_view["variant"],
    variant_view["mean_state_advantage"],
)

ax.axhline(0, linewidth=1)
ax.set_title("Notebook 11: Mean State Advantage by Variant")
ax.set_xlabel("Variant")
ax.set_ylabel("Mean Stateful - Stateless Reward")
ax.grid(True, axis="y", alpha=0.3)

advantage_variant_path = FIGURES_DIR / "11_state_advantage_by_variant.png"
fig.tight_layout()
fig.savefig(advantage_variant_path, dpi=160)

advantage_variant_path

## 8. Figure — Help / Hurt / Neutral Map

This figure counts whether state helped, hurt, or had no effect in each variant.

In [ ]:
help_hurt = variant_view[[
    "variant",
    "helped_instances",
    "hurt_instances",
    "neutral_instances",
]].set_index("variant")

fig, ax = plt.subplots(figsize=(8, 5))

bottom = np.zeros(len(help_hurt))

for column in ["helped_instances", "neutral_instances", "hurt_instances"]:
    values = help_hurt[column].values
    ax.bar(help_hurt.index, values, bottom=bottom, label=column.replace("_instances", ""))
    bottom += values

ax.set_title("Notebook 11: State Help / Neutral / Hurt Map")
ax.set_xlabel("Variant")
ax.set_ylabel("Instance Count")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)

help_hurt_path = FIGURES_DIR / "11_state_help_hurt_map.png"
fig.tight_layout()
fig.savefig(help_hurt_path, dpi=160)

help_hurt_path

## 9. Summary

Notebook 11 summarizes how much accumulated state helped across the schedule.

In [ ]:
summary = {
    "total_instances": int(len(df)),
    "state_helped_instances": int(df["state_helped"].sum()),
    "state_hurt_instances": int(df["state_hurt"].sum()),
    "state_neutral_instances": int(df["state_neutral"].sum()),
    "help_rate": float(df["state_helped"].mean()),
    "hurt_rate": float(df["state_hurt"].mean()),
    "mean_stateful_reward": float(df["stateful_reward"].mean()),
    "mean_stateless_reward": float(df["stateless_reward"].mean()),
    "mean_state_advantage": float(df["state_advantage"].mean()),
    "total_state_advantage": float(df["state_advantage"].sum()),
    "max_state_advantage": float(df["state_advantage"].max()),
    "max_state_advantage_instance": int(df.loc[df["state_advantage"].idxmax(), "instance"]),
    "min_state_advantage": float(df["state_advantage"].min()),
    "min_state_advantage_instance": int(df.loc[df["state_advantage"].idxmin(), "instance"]),
    "trajectory_source": str(source_file) if source_file is not None else "fallback",
}

summary

## 10. Export Artifacts

In [ ]:
instances_path = RESULTS_DIR / "11_state_vs_stateless_instances.csv"
variant_path = RESULTS_DIR / "11_state_vs_stateless_by_variant.csv"
summary_path = RESULTS_DIR / "11_state_vs_stateless_summary.json"
zip_path = RESULTS_DIR / "11_state_vs_stateless_artifacts.zip"

instance_view.to_csv(instances_path, index=False)
variant_view.to_csv(variant_path, index=False)

summary_with_metadata = {
    **summary,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "11_state_vs_stateless.ipynb",
    "extension": "continual-learning-bench-rml",
    "repo": REPO_URL,
}

with open(summary_path, "w") as f:
    json.dump(summary_with_metadata, f, indent=2)

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for path in [
        instances_path,
        variant_path,
        summary_path,
        reward_gap_path,
        advantage_variant_path,
        help_hurt_path,
    ]:
        z.write(path, arcname=path.name)

print("Saved artifacts:")
for path in [
    instances_path,
    variant_path,
    summary_path,
    reward_gap_path,
    advantage_variant_path,
    help_hurt_path,
    zip_path,
]:
    print("-", path)

## 11. Download Artifacts

In Colab, this downloads the zip.

Locally, the archive is saved under:

```text
rml/results/11_state_vs_stateless_artifacts.zip
```

In [ ]:
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    print("Download helper is only active in Colab.")
    print("Artifact archive:", zip_path)

## 12. Notebook 11 Claim

Notebook 11 returns to the core CL-BENCH comparison:

\[
\text{stateful} \; vs. \; \text{stateless}
\]

In RML terms:

> State is useful only when it preserves reusable context.

The distinction matters:

- memory can help,
- memory can do nothing,
- memory can hurt.

Notebook 13 will next ask whether useful structure remains stable across context changes.